# Tutorial - MultiRotor System

This is a basic tutorial on how to use the `MultiRotor` class for rotordynamics analysis. Before starting this tutorial, be sure you are already familiar with the ROSS library. What changes here is basically the part of building the model, since running the analyses will be practically the same as seen in the other tutorials.

When two shafts are joined together by gears, coupling can occur between lateral and torsional vibrations. This interaction is modeled in ROSS based on {cite:t}`rao1998theoretical` and {cite:t}`kaplan2013rotor`. In this work, a typical spur gear pair is modeled as a pair of rigid disks connected by a spring and a damper, considering the pressure angle ($\alpha$) and the orientation angle ($\varphi$) as shown below:

<div style="text-align: center;">
    <img src="../_static/img/img_tutorial4_gearmesh.png" alt="Gear mesh" style="width: 500px; height: auto;">
    <br>
    <small>Figure 1: Global coordinate system of a spur gear pair (Yang et al., 2016).</small>
</div>

As you can see, an element not yet shown is needed to model the multi-rotor system: the `GearElement` or its variation, `GearElementTVMS`. This new element resembles the `DiskElement` with added attributes, which will be shown next. If you want to use time-varying stiffness, it's recommended to use `GearElementTVMS`.


In [1]:
import ross as rs
import numpy as np
import pandas as pd

from copy import deepcopy
from ross.units import Q_

# Make sure the default renderer is set to 'notebook' for inline plots in Jupyter
import plotly.io as pio

pio.renderers.default = "notebook"

# Section 1: Constant Mesh Stiffness

## 1.1 GearElement Class

The `GearElement` class allows you to create gear elements from their mass and inertia. It is a subclass of `DiskElement` that requires information related to its pitch diameter and pressure angle.

In [2]:
rs.GearElement(
    n=0,
    m=5,
    Id=0.002,
    Ip=0.004,
    n_teeth=20,
    pitch_diameter=0.5,
    pr_angle=Q_(22.5, "deg"),
    tag="Gear",
)

GearElement(Id=0.002, Ip=0.004, m=5.0, color='Goldenrod', n=0, scale_factor=1.0, tag='Gear')

## 1.2 MultiRotor Class

`MultiRotor` is a subclass of the `Rotor` class. It takes two rotors (driving and driven) as arguments and couples them with their gears. The object created has several methods that can be used to evaluate the dynamics of the model (they all start with the `.run_` prefix).

To use this class, you must input the already instantiated rotors and each one needs at least one gear element.

The shaft elements are renumbered starting with the elements of the driving rotor.

To assemble the matrices, the driving and driven rotor matrices are joined.
For the stiffness matrix, the coupling is considered at the nodes of the gears in contact.


### 1.2.1 Creating a spur geared two-shaft rotor system
Let's create a simple model with two rotors connected by a pair of spur gears and supported by flexible bearings, as shown in Fig. 2. For more details on the description of the model, see the work of {cite:t}`rao1998theoretical`.

<div style="text-align: center;">
    <img src="../_static/img/img_tutorial4_multirotor.png" alt="MultiRotor" style="width: 500px; height: auto;">
    <br>
    <small>Figure 2: Rotors connected by a pair of spur gears (Friswell et al., 2010).</small>
</div>

In Figure 2, the first node is 1 (one), but we must remember that in ROSS the node count starts at 0 (zero).

#### 1.2.1.1 Creating material

In [3]:
# Creating material
material = rs.Material(name="mat_steel", rho=7800, E=207e9, G_s=79.5e9)

#### 1.2.1.2 Creating the driving rotor

In [4]:
# Rotor 1

L1 = [0.1, 4.24, 1.16, 0.3]
d1 = [0.3, 0.3, 0.22, 0.22]
shaft1 = [
    rs.ShaftElement(
        L=L1[i],
        idl=0.0,
        odl=d1[i],
        material=material,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(L1))
]

generator = rs.DiskElement(
    n=1,
    m=525.7,
    Id=16.1,
    Ip=32.2,
)

disk = rs.DiskElement(
    n=2,
    m=116.04,
    Id=3.115,
    Ip=6.23,
)

gear1 = rs.GearElement(
    n=4,
    m=726.4,
    Id=56.95,
    Ip=113.9,
    n_teeth=328,
    base_diameter=0.5086 * 2,
    pr_angle=Q_(22.5, "deg"),
    helix_angle=0,
)

bearing1 = rs.BearingElement(n=0, kxx=183.9e6, kyy=200.4e6, cxx=3e3)

bearing2 = rs.BearingElement(n=3, kxx=183.9e6, kyy=200.4e6, cxx=3e3)

rotor1 = rs.Rotor(
    shaft1,
    [generator, disk, gear1],
    [bearing1, bearing2],
)

rotor1.plot_rotor()

#### 1.2.1.3 Creating the driven rotor

In [5]:
# Rotor 2

L2 = [0.3, 5, 0.1]
d2 = [0.15, 0.15, 0.15]
shaft2 = [
    rs.ShaftElement(
        L=L2[i],
        idl=0.0,
        odl=d2[i],
        material=material,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(L2))
]

base_radius = rs.Q_(0.03567, "m")
pressure_angle = rs.Q_(22.5, "deg")
pitch_diameter = 2 * base_radius / np.cos(pressure_angle)

gear2 = rs.GearElement(
    n=0,
    m=5,
    Id=0.002,
    Ip=0.004,
    n_teeth=23,
    pitch_diameter=pitch_diameter,
    pr_angle=pressure_angle,
    helix_angle=0,
)

turbine = rs.DiskElement(n=2, m=7.45, Id=0.0745, Ip=0.149)

bearing3 = rs.BearingElement(n=1, kxx=10.1e6, kyy=41.6e6, cxx=3e3)

bearing4 = rs.BearingElement(n=3, kxx=10.1e6, kyy=41.6e6, cxx=3e3)

rotor2 = rs.Rotor(
    shaft2,
    [gear2, turbine],
    [bearing3, bearing4],
)

rotor2.plot_rotor()

#### 1.2.1.4 Connecting rotors

To build the multi-rotor model, we need to provide the following, in order:
- the driving rotor,
- the driven rotor,
- the tuple with the pair of coupled nodes (first number corresponds to the gear node of the driving rotor, and the second of the driven rotor),
- the gear ratio, and
- the gear mesh stiffness.

Finally, we can specify:
- the orientation angle (if not provided, zero is used by default),
- the position of the driven rotor relative to the driving rotor for visualization in the plot (`"above"` or `"below"`), and
- a tag.


In [6]:
# Creating multi-rotor

multi_rotor = rs.MultiRotor(
    rotor1,
    rotor2,
    coupled_nodes=(4, 0),
    gear_mesh_stiffness=1e8,
    orientation_angle=0,
    position="below",
)

multi_rotor.plot_rotor()

### 1.2.2 Running analyses
We will run some analyses for the multi-rotor in this section and even compare results from the literature.

#### 1.2.2.1 Modal analysis

Let's start with the modal analysis to obtain the natural frequencies for the coupled rotor when the generator runs at
1500 RPM. Then we will compare the results with {cite:t}`friswell2010dynamics`.

It is worth noting that in the analyses, we must always specify the speed of the driving rotor, not the driven one.


In [7]:
# Friswell et al. (2010) results for natural frequencies:
Friswell_results = np.array(
    [
        11.641,
        12.284,
        17.268,
        18.458,
        23.956,
        37.681,
        49.889,
        50.861,
        56.248,
        57.752,
        59.188,
        63.113,
        74.203,
    ]
)

speed = Q_(1500, "RPM")
frequencies = 13

modal = multi_rotor.run_modal(speed, num_modes=frequencies * 2)
wd = np.round(Q_(modal.wd, "rad/s").to("Hz").m, 5)

print("Natural frequencies (Hz)")
pd.DataFrame(
    {
        "Friswell et al.": Friswell_results,
        "ROSS": wd,
        "Error (%)": np.abs(wd - Friswell_results) / Friswell_results * 100,
    }
)

Natural frequencies (Hz)


,Friswell et al.,ROSS,Error (%)
0,11.641,11.64052,0.004123
1,12.284,12.28429,0.002361
2,17.268,17.26763,0.002143
3,18.458,18.45766,0.001842
4,23.956,23.95561,0.001628
5,37.681,37.68111,0.000292
6,49.889,49.88884,0.000321
7,50.861,50.86107,0.000138
8,56.248,56.24761,0.000693
9,57.752,57.75190,0.000173


#### 1.2.2.2 Campbell diagram

To obtain the Campbell diagram we can proceed in the same way as seen for a single rotor. Remember that the reference speeds / frequencies are relative to the driving rotor.

In the Campbell diagram below, the dashed lines show the shaft rotation speeds corresponding to the generator (blue, node 1) and turbine (yellow, node 7).


In [8]:
frequency_range = Q_(np.arange(0, 5000, 100), "RPM")

gear_ratio = multi_rotor.mesh.gear_ratio

campbell = multi_rotor.run_campbell(frequency_range, frequencies=13)
campbell.plot(frequency_units="Hz", harmonics=[1, round(gear_ratio, 3)]).show()

In [9]:
nodes = [2, 7]
unb_mag = [35.505e-3, 0.449e-3]
unb_phase = [0, 0]

dt = 1e-3
t = np.arange(0, 1200, dt)
speed1 = Q_(5000, "RPM").to_base_units().m  # Generator rotor speed

# Unbalance force
F = multi_rotor.unbalance_force_over_time(nodes, unb_mag, unb_phase, speed1, t)

In [10]:
# Time response
time_resp = multi_rotor.run_time_response(speed1, F.T, t)
amp_resp = time_resp.yout

Due to computational cost limitations, the entire analysis performed for comparison is not included in this tutorial. However, by running the `.run_time_response` method across the complete range of speeds, it is possible to obtain the following responses at nodes 2 and 7 as shown in the figures below:

<div style="text-align: center;">
    <img src="../_static/img/img_tutorial4_node2.png" alt="MultiRotor" style="width: 100%; max-width: 720px; height: auto;">
    <img src="../_static/img/img_tutorial4_node7.png" alt="MultiRotor" style="width: 100%; max-width: 720px; height: auto;">
    <br>
    <small>Figure 3: Comparison of ROSS results with Yang et al. (2016) for the spur geared two-shaft rotor system.</small>
</div>

### 1.2.3 Creating a spur geared multi-shaft rotor system

Let's create a more complex model with three connected rotors. More details of this example can be found in {cite:t}`yang2016general`.


In [11]:
# Rotor 1

L1 = [300, 92, 200, 200, 92, 300]
r1 = [61.5, 75, 75, 75, 75, 61.5]
shaft1_coupled = [
    rs.ShaftElement(
        L=L1[i] * 1e-3,
        idl=0.0,
        odl=r1[i] * 2e-3,
        material=material,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(L1))
]

D1 = rs.DiskElement(
    n=0,
    m=66.63,
    Id=0.431,
    Ip=0.735,
)

D2 = rs.DiskElement(
    n=6,
    m=69.83,
    Id=0.542,
    Ip=0.884,
)

cxx = 3e3
B1 = rs.BearingElement(n=2, kxx=5.5e8, kyy=6.7e8, cxx=cxx)
B2 = rs.BearingElement(n=4, kxx=5.5e8, kyy=6.7e8, cxx=cxx)

pressure_angle = Q_(22.5, "deg")

G1 = rs.GearElement(
    n=3,
    m=14.37,
    Id=0.068,
    Ip=0.136,
    n_teeth=37,
    base_diameter=0.19,
    pr_angle=pressure_angle,
)

rotor1 = rs.Rotor(
    shaft1_coupled,
    [D1, D2, G1],
    [B1, B2],
)

# Rotor 2

L2 = [80, 200, 200, 640]
r2 = [160.5, 160.5, 130.5, 130.5]
shaft2 = [
    rs.ShaftElement(
        L=L2[i] * 1e-3,
        idl=0.0,
        odl=r2[i] * 2e-3,
        material=material,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(L2))
]

B3 = rs.BearingElement(n=1, kxx=3.2e9, kyy=4.6e9, cxx=cxx)
B4 = rs.BearingElement(n=3, kxx=3.2e9, kyy=4.6e9, cxx=cxx)

G2 = rs.GearElement(
    n=2,
    m=813.79,
    Id=52.36,
    Ip=104.72,
    n_teeth=244,
    base_diameter=1.23,
    pr_angle=pressure_angle,
)

rotor2 = rs.Rotor(
    shaft2,
    [G2],
    [B3, B4],
)

# Rotor 3

L3 = [300, 110, 200, 200, 110, 300]
r3 = [64.5, 76.5, 76.5, 76.5, 76.5, 64.5]
shaft3 = [
    rs.ShaftElement(
        L=L3[i] * 1e-3,
        idl=0.0,
        odl=r3[i] * 2e-3,
        material=material,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    for i in range(len(L1))
]

D3 = rs.DiskElement(
    n=0,
    m=95.06,
    Id=1.097,
    Ip=1.532,
)

D4 = rs.DiskElement(
    n=6,
    m=96.22,
    Id=1.110,
    Ip=1.620,
)

G3 = rs.GearElement(
    n=3,
    m=19.52,
    Id=0.098,
    Ip=0.195,
    n_teeth=47,
    base_diameter=0.24,
    pr_angle=pressure_angle,
)

B5 = rs.BearingElement(n=2, kxx=7.2e8, kyy=8.4e8, cxx=cxx)
B6 = rs.BearingElement(n=4, kxx=7.2e8, kyy=8.4e8, cxx=cxx)

rotor3 = rs.Rotor(
    shaft3,
    [D3, D4, G3],
    [B5, B6],
)

In [12]:
# Connect rotor 1 with rotor 2 (driving rotor)
multi_rotor1 = rs.MultiRotor(
    rotor2,
    rotor1,
    coupled_nodes=(2, 3),
    gear_mesh_stiffness=2.55e8,
    orientation_angle=Q_(270, "deg"),
    position="above",
)

In [13]:
# Connect rotor 3 with rotor 2 in multi rotor
psi = 90
final_system = rs.MultiRotor(
    multi_rotor1,
    rotor3,
    coupled_nodes=(2, 3),
    gear_mesh_stiffness=2.6e8,
    orientation_angle=Q_(270 - psi, "deg"),
    position="below",
)

In [14]:
final_system.plot_rotor()

If we apply two unbalances of 92 g·mm in phase opposition at the two ends of rotor 1 (nodes 5 and 11), and one unbalance of 294 g·mm at the middle of rotor 3 (node 15), we can obtain the following response at node 15:


<div style="text-align: center;">
    <img src="../_static/img/img_tutorial4_multi_compared.png" alt="MultiRotor" style="width: 100%; max-width: 720px; height: auto;">
    <br>
    <small>Figure 4: Comparison of ROSS results with Yang et al. (2016) for the spur geared multi-shaft rotor system.</small>
</div>


We can also vary the orientation angle between the rotors:

<div style="text-align: center;">
    <img src="../_static/img/img_tutorial4_multi_all.png" alt="MultiRotor" style="width: 100%; max-width: 720px; height: auto;">
    <br>
    <small>Figure 5: Unbalance responses at node 15 with three orientation angles.</small>
</div>

# Section 2: Time-Varying Mesh Stiffness

The time-varying mesh stiffness method is used to capture the fluctuation in gear mesh stiffness over time. This variation results from factors such as the complete tooth profile geometry and the contact ratio of the gear mesh, modeled according to {cite:t}`ma2014time`.

## 2.1 GearElementTVMS Class

The `GearElementTVMS` class allows you to create gear elements that include time-varying mesh stiffness (TVMS) calculations.

To create a `GearElementTVMS` object, the following design parameters are required:

- `material` : Gear's construction material

- `width` : Tooth width

- `bore_diameter` : Inner diameter

- `module` : Gear module

- `n_teeth` : Number of teeth

- `pr_angle` : Normal pressure angle (Default: 20 deg)

- `helix_angle` : Helix angle for helical gears (Default: 0.0)

- `addendum_coeff` : Addendum coefficient (Default: 1.0)

- `tip_clearance_coeff` : Gear's clearance coefficient (Default: 0.25)

In [15]:
rs.GearElementTVMS(
    n=0,
    material=rs.materials.steel,
    width=0.03,
    bore_diameter=0.05,
    module=0.005,
    n_teeth=30,
    pr_angle=Q_(22.5, "deg"),
    helix_angle=0,
    addendum_coeff=1,
    tip_clearance_coeff=0.25,
)

GearElementTVMS(Id=0.0060266, Ip=0.011501, m=3.6804, color='Goldenrod', n=0, scale_factor=1.0, tag=None)

## 2.2 MultiRotor with Updated Stiffness

### 2.2.1 Creating model
Let's recreate the two rotors from Section 1.2.1, but using the `GearElementTVMS` instead.

#### 2.2.1.1 Defining common parameters

In [16]:
rho = Q_(7850, "kg/m**3")
steel = rs.Material(name="Steel", rho=rho, E=2e11, Poisson=0.3)
helix_angle = Q_(0, "deg")
pressure_angle = Q_(22.5, "deg")

#### 2.2.1.2 Creating driving gear

In [17]:
bore_diameter = Q_(0.22, "m")  # Shaft outer diameter = Gear internal diameter
m = Q_(726.4, "kg")
Ip = Q_(113.9, "kg*m**2")

pitch_diameter = np.sqrt((8 * Ip) / (m) - bore_diameter**2)  # Gear outer diameter

width = (4 * m) / (np.pi * rho * (pitch_diameter**2 - bore_diameter**2))

n_teeth = 328
module = pitch_diameter / n_teeth

gear1 = rs.GearElementTVMS(
    n=4,
    material=steel,
    width=width,
    bore_diameter=bore_diameter,
    module=module,
    n_teeth=n_teeth,
    pr_angle=pressure_angle,
    helix_angle=helix_angle,
)

#### 2.2.1.3 Creating driven gear

In [18]:
bore_diameter = Q_(0.015, "m")
m = Q_(5, "kg")
base_radius = Q_(0.03567, "m")

pitch_diameter = 2 * base_radius / np.cos(pressure_angle)

n_teeth = 23
module = pitch_diameter / n_teeth

gear2 = rs.GearElementTVMS(
    n=0,
    material=steel,
    width=width,
    bore_diameter=bore_diameter,
    module=module,
    n_teeth=n_teeth,
    pr_angle=pressure_angle,
    helix_angle=helix_angle,
)

#### 2.2.1.4 Recreating rotors

In [19]:
rotor1 = rs.Rotor(
    shaft1,
    [generator, disk, gear1],
    [bearing1, bearing2],
)

rotor2 = rs.Rotor(
    shaft2,
    [gear2, turbine],
    [bearing3, bearing4],
)

### 2.2.2 Defining the default profile for gear mesh stiffness
To obtain the default gear mesh stiffness profile, use `GearElementTVMS` with the `MultiRotor` flag `update_mesh_stiffness` set to `True`.

In [20]:
multi_rotor2 = rs.MultiRotor(
    rotor1,
    rotor2,
    coupled_nodes=(4, 0),
    update_mesh_stiffness=True,
    position="below",
    orientation_angle=0,
)

/home/raphaelts/ross/ross/multi_rotor/gear_element.py:853: UserWarning: Extrapolating gear body coefficients described by Sainsot et al. (2014). Be careful when post-processing the results.
  warn(


#### 2.2.2.1 Plotting stiffness profile
To plot the stiffness profile, use `MultiRotor.mesh.plot_stiffness_profile` and specify the number of mesh periods to plot.

In [21]:
multi_rotor2.mesh.plot_stiffness_profile(n_mesh_period=2)

#### 2.2.2.2 Extracting parameters
You can obtain the average gear mesh stiffness and contact ratio using `.mesh.stiffness` and `.mesh.contact_ratio`, respectively.

In [22]:
stiff_avg = multi_rotor2.mesh.stiffness
print(f"Stiffness Average = {stiff_avg:.2e} N/m")

contact_ratio = multi_rotor2.mesh.contact_ratio
print(f"Contact Ratio = {contact_ratio:.2f}")

Stiffness Average = 1.85e+09 N/m
Contact Ratio = 1.64


### 2.2.3 Defining the rectangular profile for gear mesh stiffness
To obtain the rectangular gear mesh stiffness profile, set the `MultiRotor` flag `update_mesh_stiffness` to `True` and provide `square_varying_stiffness`, a dictionary with `"enable"` set to `True` and an `"amplitude_ratio"` value representing the amplitude ratio relative to the mean value.

In [23]:
multi_rotor3 = rs.MultiRotor(
    rotor1,
    rotor2,
    coupled_nodes=(4, 0),
    update_mesh_stiffness=True,
    square_varying_stiffness={"enable": True, "amplitude_ratio": 0.225},
    position="below",
    orientation_angle=0,
)

#### 2.2.3.1 Plotting stiffness profile
To plot the stiffness profile, use `MultiRotor.mesh.plot_stiffness_profile` and specify the number of mesh periods to plot.


In [24]:
multi_rotor3.mesh.plot_stiffness_profile(n_mesh_period=2)

#### 2.2.3.2 Extracting parameters
You can obtain the average gear mesh stiffness and contact ratio using `.mesh.stiffness` and `.mesh.contact_ratio`, respectively.


In [25]:
stiff_avg = multi_rotor3.mesh.stiffness
print(f"Stiffness Average = {stiff_avg:.2e} N/m")

contact_ratio = multi_rotor3.mesh.contact_ratio
print(f"Contact Ratio = {contact_ratio:.2f}")

Stiffness Average = 1.85e+09 N/m
Contact Ratio = 1.64


# Section 3: Backlash Analysis

This section describes the configuration and execution of an analysis that considers **_backlash (dynamic clearance)_** between gears, as well as the dynamic variation of gear mesh parameters. The **_backlash_** introduces a strongly nonlinear behavior to the system. 

The implementation and parameters presented here are based on and validated against {cite:t}`yi2019nonlinear`, with more details discussed in {cite:t}`murillo2026implementation`.

In the `MultiRotor` class, dynamic backlash is activated by passing a `backlash` dictionary during instantiation. The main configurable parameters are:

- `enable` : Enables or disables backlash consideration in the simulation (`True` or `False`)

- `initial_value` : The nominal initial clearance value

- `error_amp` : Amplitude of the gear manufacturing/assembly error

- `smooth_operator` : Defines whether a smooth transition function will be used to avoid numerical discontinuities in contact forces (`True` or `False`)

- `sigma` : Slope/smoothing parameter used when `smooth_operator` is active


>**Note:** When activating *backlash*, mesh stiffness will always be updated. 

## 3.1 MultiRotor with Backlash
This simple example from {cite:t}`yi2019nonlinear` uses only `GearElementTVMS` for the analysis. However, rotor mounting is still necessary, done below using an insignificant `ShaftElement` instance.

In [26]:
steel = rs.Material(name="Steel", rho=7850, E=2e11, Poisson=0.3)
steel_stiff = rs.Material(name="Steel_Stiff", rho=0.01, E=1e15, Poisson=0.3)
shaft = rs.ShaftElement(n=0, L=0.0001, idl=0.0, odl=0.0001, material=steel_stiff)

kxx = kyy = 1.0e8
cxx = cyy = 512.64
bearing = rs.BearingElement(n=0, kxx=kxx, kyy=kyy, cxx=cxx, cyy=cyy)

n_teeth = 20
module = 0.01
pitch_diam = module * n_teeth
width = 0.030
m = 6.57

gear = rs.GearElementTVMS(
    n=0,
    material=steel,
    width=width,
    bore_diameter=np.sqrt(pitch_diam**2 - (4 * m) / (np.pi * width * steel.rho)),
    module=module,
    n_teeth=n_teeth,
    pr_angle=rs.Q_(20.0, "deg"),
    helix_angle=0,
    addendum_coeff=1,
    tip_clearance_coeff=0.25,
)

rotor1 = rs.Rotor(
    shaft_elements=[shaft], disk_elements=[gear], bearing_elements=[bearing]
)

rotor2 = deepcopy(rotor1)  # Rotor 2 is identical to Rotor 1

mr_backlash = rs.MultiRotor(
    driving_rotor=rotor1,
    driven_rotor=rotor2,
    coupled_nodes=(0, 0),
    square_varying_stiffness={"enable": True, "amplitude_ratio": 0.275},
    backlash={
        "enable": True,
        "initial_value": 5e-5,
        "error_amp": 2e-5,
        "smooth_operator": False,
        "sigma": 1e5,
    },
    orientation_angle=0.0,
    position="above",
)

## 3.2 Time Response Simulation

Due to the strong nonlinearity introduced by the gear mesh with *backlash*, the analysis must be performed in the time domain using the `.run_time_response` method.

Excitation torques, $T = T_0 + T_a \sin(\omega t)$, are applied to the torsional degrees of freedom of the gears, and the integration is executed using the Newmark method with a more robust solver to handle the nonlinearities.

**Consideration for simulation speed:** At $1000\ \text{RPM}$ ($16.67\ \text{Hz}$), the mesh period $T_m$ is small compared to the total simulation time ($t_f = 2.65\ \text{s}$), requiring an adequate number of points per mesh cycle (e.g., 6000 points per cycle) to ensure stability and convergence of the numerical solution.

In [27]:
T10, T1a = 300.0, 100.0
T20, T2a = 300.0, 100.0

speed = Q_(1000, "RPM").to("rad/s").m
Tm = 2 * np.pi / speed

tf = 2.65
n_cycles = int(np.ceil(tf / Tm))
n_points = 6000

t = np.linspace(0, n_cycles * Tm, n_cycles * n_points)

nodes = [int(e.n) for e in mr_backlash.disk_elements if isinstance(e, rs.GearElement)]

w1 = speed
w2 = mr_backlash.mesh.gear_ratio * w1
num_dof = mr_backlash.number_dof

# Force/torque excitation matrix
F = np.zeros((len(t), mr_backlash.ndof))
F[:, nodes[0] * num_dof + 5] = T10 + T1a * np.sin(w1 * t)
F[:, nodes[1] * num_dof + 5] = T20 + T2a * np.sin(w2 * t)

time_results = mr_backlash.run_time_response(
    speed=speed, t=t, F=F, method="newmark", newmark_type="robust"
)

Running direct method


## 3.3 Analysis of Results

The results of the gear mesh contact dynamics are stored in the `mesh_dynamics` attribute of the returned object. Among the available results are:

- `transmission_error` : Dynamic transmission error

- `backlash` : Instantaneous backlash limit, considering fluctuations and deformations

- `mesh_force` : Dynamic mesh force

- `mesh_stiffness` : Time-varying mesh stiffness

- `center_distance` : Dynamic center distance

- `pressure_angle` : Dynamic pressure angle

- `contact_ratio` : Contact ratio

In [28]:
mesh_results = time_results.mesh_dynamics

print(f"Mean Transmission Error: {np.mean(mesh_results['transmission_error']):.5e} m")
print(f"Mean Backlash: {np.mean(mesh_results['backlash']):.5e} m")
print(f"Mean Mesh Force: {np.mean(mesh_results['mesh_force']):.2f} N")
print(f"Mean Mesh Stiffness: {np.mean(mesh_results['mesh_stiffness']):.2f} N/m")
print(f"Mean Center Distance: {np.mean(mesh_results['center_distance']):.5f} m")
print(f"Mean Pressure Angle: {np.mean(mesh_results['pressure_angle']):.5f} rad")
print(f"Mean Contact Ratio: {np.mean(mesh_results['contact_ratio']):.5f}")

Mean Transmission Error: 7.15688e-05 m
Mean Backlash: 6.49805e-05 m
Mean Mesh Force: 3192.47 N
Mean Mesh Stiffness: 517905171.91 N/m
Mean Center Distance: 0.20004 m
Mean Pressure Angle: 0.34967 rad
Mean Contact Ratio: 1.55251


### 3.3.1 Plotting parameters

In addition to the common `TimeResponseResults` plots, each parameter can be plotted individually using its corresponding `.plot_` method (e.g., `.plot_mesh_force`, `.plot_backlash`), or all parameters can be viewed together in a single overview using `.plot_dashboard`.

In [29]:
time_results.plot_dashboard(
    time_range=(2, 2.12),
    frequency_range=rs.Q_((0, 400000), "RPM"),
    frequency_units="RPM",
)

### 3.3.2 Validating results
Results were compared against {cite:t}`yi2019nonlinear`, as shown in the figures below.

In [30]:
fig_DTE = time_results.plot_transmission_error(time_range=(2.4, 2.52), data_units="μm")

<div style="text-align: center;">
    <img src="../_static/img/img_tutorial4_dte_1000rpm.svg" alt="MultiRotor Backlash DTE" style="width: auto; height: auto;">
    <br>
    <small>Figure 6: Comparison of ROSS results with Yi et al. (2019) for transmisson error (1000 RPM).</small>
</div>

In [31]:
fig_bt = time_results.plot_backlash(time_range=(2.4, 2.6), data_units="μm")

<div style="text-align: center;">
    <img src="../_static/img/img_tutorial4_bt_1000rpm.svg" alt="MultiRotor Backlash bt" style="width: auto; height: auto;">
    <br>
    <small>Figure 7: Comparison of ROSS results with Yi et al. (2019) for backlash clearance (1000 RPM).</small>
</div>

# References

```{bibliography}
:filter: docname in docnames
```